# Лиги экономии — движок, лиги, экономика, антифрод

1. **Движок очков** — вогнутая функция суммы, дневной потолок, активные дни и недели, серия.
2. **Формирование групп** — подбор похожих покупателей, размер группы зависит от лиги.
3. **Месячный прогон** — переходы между лигами, сезонное окно, паузы.
4. **Экономика** — стоимость награды против прироста маржи по лигам, поиск границы окупаемости.
5. **Антифрод** — семь объяснимых признаков, порог, precision на размеченных данных.

**Про числа.** Все параметры поведения и экономики — допущения, собранные в конфигах ниже.
Реальными являются только доли сегментов по трём сетям, заложенные в генератор.

In [1]:
import warnings, math, json
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 0. Загрузка данных

In [4]:
DATA = Path('./Generate')
print("Данные:", DATA.resolve())

profiles = pd.read_csv(DATA / "profiles.csv")
receipts = pd.read_csv(DATA / "receipts.csv.gz")
stores = pd.read_csv(DATA / "stores.csv")

receipts["ts"] = pd.to_datetime(receipts["date"] + " " + receipts["time"])
receipts["month"] = receipts["date"].str.slice(0, 7)
receipts = receipts.merge(stores[["store_id", "city"]], on="store_id", how="left")

print(f"Профилей: {len(profiles):,}   чеков: {len(receipts):,}   магазинов: {len(stores):,}")
print("Месяцы:", sorted(receipts["month"].unique()))

Данные: C:\Users\Константин\OneDrive\Desktop\Проектирование системы игровой лояльности\Generate
Профилей: 8,000   чеков: 416,230   магазинов: 6,665
Месяцы: ['2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08']


## 2. Движок очков

Все правила зачёта вынесены в один конфиг. 

Меняем здесь — пересчитывается всё остальное.

**Правила зачёта чека.** Чек ниже минимальной суммы не учитывается вообще. Второй чек в том же
магазине в тот же день засчитывается только если прошёл минимальный интервал — это и есть
контрмера против дробления, встроенная в механику, вместо скоринга.

**Дневной потолок.** Всё, что человек потратил сверх потолка за день, в очки не идёт.
Ломает стратегию «купить на месяц вперёд одним чеком и выиграть месяц».

**Вогнутость.** Очки за сумму считаются как `log1p(сумма / масштаб)`: первые рубли дают
заметный прирост, каждый следующий — меньше.

In [5]:
POINTS = {
    "MIN_RECEIPT_RUB": 100,       # чек ниже не засчитывается
    "MIN_GAP_MIN": 40,            # минимальный интервал между зачитываемыми чеками в одном магазине за день
    "DAILY_CAP_RUB": 2500,        # дневной потолок засчитываемой суммы
    "SUM_SCALE_RUB": 4000,        # масштаб вогнутой функции
    "W_SUM": 0.5,                 # вес суммы
    "W_REG": 0.5,                 # вес регулярности активности
    "REG_DAY": 1.0,               # вклад активного дня
    "REG_WEEK": 2.0,              # вклад активной недели
    "REG_STREAK": 1.5,            # вклад каждой недели в текущей серии
    "TOTAL_SCALE": 500,           # общий масштаб очков, для наглядности
}
assert abs(POINTS["W_SUM"] + POINTS["W_REG"] - 1) < 1e-9

In [6]:
def qualify(rec: pd.DataFrame, cfg=POINTS) -> pd.DataFrame:
    """Отбор чеков, которые идут в зачёт. Возвращает копию с колонкой qualified."""
    r = rec.sort_values(["profile_id", "store_id", "ts"]).copy()
    r["qualified"] = r["amount_rub"] >= cfg["MIN_RECEIPT_RUB"]

    # интервал до предыдущего чека того же человека в том же магазине в тот же день
    key = [r["profile_id"], r["store_id"], r["date"]]
    prev = r.groupby(key, sort=False)["ts"].shift(1)
    gap_min = (r["ts"] - prev).dt.total_seconds() / 60
    too_close = gap_min.notna() & (gap_min < cfg["MIN_GAP_MIN"])
    r.loc[too_close, "qualified"] = False
    r["gap_min"] = gap_min
    return r

rq = qualify(receipts)
print(f"Засчитано чеков: {rq['qualified'].sum():,} из {len(rq):,} "
      f"({100*rq['qualified'].mean():.1f} %)")
print(f"  отсеяно по минимальной сумме: {(~(receipts['amount_rub'] >= POINTS['MIN_RECEIPT_RUB'])).sum():,}")
print(f"  отсеяно по интервалу:        {(rq['gap_min'].notna() & (rq['gap_min'] < POINTS['MIN_GAP_MIN'])).sum():,}")

Засчитано чеков: 406,108 из 416,230 (97.6 %)
  отсеяно по минимальной сумме: 6,107
  отсеяно по интервалу:        4,950


In [7]:
def monthly_points(rq: pd.DataFrame, cfg=POINTS) -> pd.DataFrame:
    """Очки за месяц: сумма с потолком и вогнутостью + регулярность."""
    q = rq[rq["qualified"]].copy()

    # дневная сумма с потолком
    daily = q.groupby(["profile_id", "month", "date"], as_index=False)["amount_rub"].sum()
    daily["capped"] = daily["amount_rub"].clip(upper=cfg["DAILY_CAP_RUB"])

    m = daily.groupby(["profile_id", "month"]).agg(
        capped_sum=("capped", "sum"),
        raw_sum=("amount_rub", "sum"),
        active_days=("date", "nunique"),
    ).reset_index()

    # активные недели и серия
    q["iso_week"] = q["ts"].dt.isocalendar().week.astype(int)
    q["iso_year"] = q["ts"].dt.isocalendar().year.astype(int)
    weeks = q.groupby(["profile_id", "month"])[["iso_year", "iso_week"]].apply(
        lambda d: len(set(zip(d["iso_year"], d["iso_week"])))
    ).rename("active_weeks").reset_index()
    m = m.merge(weeks, on=["profile_id", "month"], how="left")

    # серия недель подряд по всей истории профиля
    wk = q[["profile_id", "iso_year", "iso_week", "month"]].drop_duplicates()
    wk["wk_idx"] = wk["iso_year"] * 53 + wk["iso_week"]
    wk = wk.sort_values(["profile_id", "wk_idx"])
    wk["gap"] = wk.groupby("profile_id")["wk_idx"].diff().fillna(99)
    wk["new_streak"] = (wk["gap"] != 1).cumsum()
    wk["streak"] = wk.groupby(["profile_id", "new_streak"]).cumcount() + 1
    streak = wk.groupby(["profile_id", "month"])["streak"].max().rename("streak_weeks").reset_index()
    m = m.merge(streak, on=["profile_id", "month"], how="left").fillna({"streak_weeks": 0})

    # очки
    raw_sum_pts = np.log1p(m["capped_sum"] / cfg["SUM_SCALE_RUB"])
    raw_reg_pts = (cfg["REG_DAY"] * m["active_days"]
                   + cfg["REG_WEEK"] * m["active_weeks"]
                   + cfg["REG_STREAK"] * m["streak_weeks"])

    # нормируем обе части к среднему 1, чтобы веса означали именно доли вклада
    m["points_sum"] = cfg["W_SUM"] * raw_sum_pts / raw_sum_pts.mean() * cfg["TOTAL_SCALE"]
    m["points_reg"] = cfg["W_REG"] * raw_reg_pts / raw_reg_pts.mean() * cfg["TOTAL_SCALE"]
    m["points"] = (m["points_sum"] + m["points_reg"]).round(1)
    return m

mp = monthly_points(rq)
mp = mp.merge(profiles[["profile_id", "segment", "network", "city_type",
                        "base_freq_per_month", "avg_check_rub", "tenure_months",
                        "is_fraud", "fraud_type"]], on="profile_id", how="left")

share_sum = mp["points_sum"].sum() / mp["points"].sum()
print(f"Строк «профиль × месяц»: {len(mp):,}")
print(f"Фактическая доля вклада суммы в очки: {100*share_sum:.1f} % (задано {100*POINTS['W_SUM']:.0f} %)")
mp[["points", "points_sum", "points_reg", "capped_sum", "active_days", "active_weeks", "streak_weeks"]].describe().round(1)

Строк «профиль × месяц»: 47,473
Фактическая доля вклада суммы в очки: 50.0 % (задано 50 %)


,points,points_sum,points_reg,capped_sum,active_days,active_weeks,streak_weeks
count,47473.0,47473.0,47473.0,47473.0,47473.0,47473.0,47473.0
mean,500.0,250.0,250.0,7955.2,8.5,3.9,5.2
std,223.9,129.2,112.6,6891.8,5.4,1.2,3.5
min,52.9,6.4,46.5,100.1,1.0,1.0,1.0
25%,331.7,150.7,160.2,3147.0,5.0,3.0,3.0
50%,489.6,236.3,248.0,5936.3,7.0,4.0,5.0
75%,655.7,336.3,325.6,10601.2,11.0,5.0,6.0
max,1386.8,744.5,739.0,66305.8,31.0,6.0,28.0


In [10]:
# на что режет потолок и как выглядит вогнутость
lost = (mp["raw_sum"] - mp["capped_sum"]).sum() / mp["raw_sum"].sum()
print(f"Доля оборота, срезанная дневным потолком: {100*lost:.1f} %")

probe = pd.DataFrame({"сумма за месяц, ₽": [2000, 5000, 10000, 20000, 40000, 80000]})
probe["очки за сумму"] = (np.log1p(probe["сумма за месяц, ₽"] / POINTS["SUM_SCALE_RUB"])
                          / np.log1p(mp["capped_sum"].mean() / POINTS["SUM_SCALE_RUB"])
                          * POINTS["W_SUM"] * POINTS["TOTAL_SCALE"]).round(0)
probe["прирост к предыдущей строке"] = probe["очки за сумму"].diff().fillna(0)
probe

Доля оборота, срезанная дневным потолком: 2.8 %


,"сумма за месяц, ₽",очки за сумму,прирост к предыдущей строке
0,2000,93.0,0.0
1,5000,185.0,92.0
2,10000,286.0,101.0
3,20000,409.0,123.0
4,40000,548.0,139.0
5,80000,695.0,147.0


## 3. Формирование групп

Группа собирается заново каждый месяц из участников одной лиги. Подбор — по похожести:
сегмент, тип населённого пункта, базовая частота и порядок среднего чека.

Размер группы зависит от лиги: 30 внизу, 12 в алмазной. Внизу лестница тянет вверх,
наверху удержание дороже роста.

In [11]:
# лига: (название, размер группы, вверх, остаются, вниз)
LEAGUES = [
    ("Бронзовая",     30, 8, 22, 0),
    ("Серебряная",    30, 7, 18, 5),
    ("Золотая",       30, 7, 16, 7),
    ("Платиновая",    28, 6, 14, 8),
    ("Рубиновая",     26, 5, 12, 9),
    ("Изумрудная",    24, 5, 10, 9),
    ("Сапфировая",    22, 4,  8, 10),
    ("Аметистовая",   20, 4,  7, 9),
    ("Обсидиановая",  18, 3,  5, 10),
    ("Жемчужная",     15, 2,  4, 9),
    ("Алмазная",      12, 0,  4, 8),
]
LEAGUE_NAMES = [x[0] for x in LEAGUES]
for name, size, up, stay, down in LEAGUES:
    assert up + stay + down == size, name
print(pd.DataFrame(LEAGUES, columns=["лига", "группа", "вверх", "остаются", "вниз"]).to_string(index=False))

        лига  группа  вверх  остаются  вниз
   Бронзовая      30      8        22     0
  Серебряная      30      7        18     5
     Золотая      30      7        16     7
  Платиновая      28      6        14     8
   Рубиновая      26      5        12     9
  Изумрудная      24      5        10     9
  Сапфировая      22      4         8    10
 Аметистовая      20      4         7     9
Обсидиановая      18      3         5    10
   Жемчужная      15      2         4     9
    Алмазная      12      0         4     8


In [13]:
def similarity_key(df: pd.DataFrame) -> pd.Series:
    """Ключ похожести: сегмент, тип города, квартиль частоты, квартиль чека."""
    def quartile(s: pd.Series) -> pd.Series:
        if len(s) < 8 or s.nunique() < 4:
            return pd.Series(0, index=s.index)
        return pd.qcut(s, 4, labels=False, duplicates="drop").fillna(0).astype(int)

    fq = quartile(df["base_freq_per_month"])
    ck = quartile(df["avg_check_rub"])
    return (df["segment"].astype(str) + "|" + df["city_type"].astype(str)
            + "|f" + fq.astype(str) + "|c" + ck.astype(str))

def build_groups(month_df: pd.DataFrame, league: int, rng) -> pd.DataFrame:
    """Разбиение участников одной лиги на группы нужного размера."""
    size = LEAGUES[league][1]
    d = month_df.copy()
    d["skey"] = similarity_key(d)
    out, gid = [], 0
    for _, chunk in d.groupby("skey", sort=False):
        idx = rng.permutation(len(chunk))
        chunk = chunk.iloc[idx]
        for start in range(0, len(chunk), size):
            part = chunk.iloc[start:start + size].copy()
            part["group_id"] = f"L{league}-{gid:05d}"
            gid += 1
            out.append(part)
    if not out:
        return d.assign(group_id=f"L{league}-empty")
    res = pd.concat(out, ignore_index=True)

    counts = res["group_id"].value_counts()
    tiny = counts[counts < max(3, size // 3)].index
    if len(tiny) > 1:
        pool = res[res["group_id"].isin(tiny)].copy()
        rest = res[~res["group_id"].isin(tiny)]
        pool = pool.iloc[rng.permutation(len(pool))]
        labels = [f"L{league}-p{i // size:04d}" for i in range(len(pool))]
        if len(pool) % size and len(pool) > size and len(pool) % size < max(3, size // 3):
            tail = len(pool) % size
            labels[-tail:] = [f"L{league}-p{(len(pool) - tail - 1) // size:04d}"] * tail
        pool["group_id"] = labels
        res = pd.concat([rest, pool], ignore_index=True)
    return res

rng = np.random.default_rng(20260903)
demo_month = mp[mp["month"] == sorted(mp["month"].unique())[1]]
demo = build_groups(demo_month, league=4, rng=rng)
sizes = demo["group_id"].value_counts()
print(f"Участников: {len(demo):,}   групп: {demo['group_id'].nunique():,}")
print(f"Размер группы: медиана {sizes.median():.0f}, минимум {sizes.min()}, максимум {sizes.max()}")
print()
print("Однородность внутри групп — стандартное отклонение среднего чека:")
print(f"  внутри групп: {demo.groupby('group_id')['avg_check_rub'].std().mean():,.0f} ₽")
print(f"  по всей лиге: {demo['avg_check_rub'].std():,.0f} ₽")

Участников: 7,897   групп: 368
Размер группы: медиана 26, минимум 8, максимум 26

Однородность внутри групп — стандартное отклонение среднего чека:
  внутри групп: 144 ₽
  по всей лиге: 412 ₽


## 4. Месячный прогон

Первый месяц истории уходит на определение стартовой лиги — новичка не бросаем в бронзовую,
если по истории он тянет выше. Дальше пять полных циклов с переходами.

Правила: понижение всегда на одну ступень; нулевая активность за месяц — понижение
независимо от места; в бронзовой вылета нет, в алмазной нет повышения.

Сезонное окно из шести слотов и паузы за 12 активных недель считаются по ходу.

In [14]:
def start_league(first_month: pd.DataFrame, n_start_max=6) -> pd.Series:
    """Стартовая лига по истории: перцентиль очков первого месяца."""
    pct = first_month["points"].rank(pct=True)
    return (pct * (n_start_max + 1)).astype(int).clip(0, n_start_max)

def zones_for(league: int, n: int):
    """Границы зон, пропорционально пересчитанные под фактический размер группы."""
    _, size, up, stay, down = LEAGUES[league]
    k = n / size
    up_n = int(round(up * k))
    down_n = int(round(down * k))
    up_n = min(up_n, max(0, n - 1))
    down_n = min(down_n, max(0, n - up_n))
    return up_n, down_n

def run_simulation(mp: pd.DataFrame, seed=20260903):
    months = sorted(mp["month"].unique())
    rng = np.random.default_rng(seed)

    state = mp[mp["month"] == months[0]][["profile_id"]].copy()
    state["league"] = start_league(mp[mp["month"] == months[0]]).values
    state["season"] = 0      # знаков в окне
    state["window"] = [[] for _ in range(len(state))]
    state["pauses"] = 0
    state["streak_for_pause"] = 0

    history = []
    for m in months[1:]:
        cur = mp[mp["month"] == m][["profile_id", "points", "active_days",
                                    "active_weeks", "streak_weeks", "capped_sum"]]
        step = state.merge(cur, on="profile_id", how="left")
        step["points"] = step["points"].fillna(0)
        step["active_days"] = step["active_days"].fillna(0)
        step["active_weeks"] = step["active_weeks"].fillna(0)
        step["capped_sum"] = step["capped_sum"].fillna(0)
        step["month"] = m

        moves = []
        for lg, part in step.groupby("league"):
            grouped = build_groups(part.merge(
                profiles[["profile_id", "segment", "city_type",
                          "base_freq_per_month", "avg_check_rub"]],
                on="profile_id", how="left"), lg, rng)
            grouped["rank"] = grouped.groupby("group_id")["points"].rank(
                ascending=False, method="first")
            gsize = grouped.groupby("group_id")["profile_id"].transform("size")
            zone = []
            for gid, g in grouped.groupby("group_id"):
                up_n, down_n = zones_for(lg, len(g))
                r = g["rank"]
                z = np.where(r <= up_n, "вверх",
                     np.where(r > len(g) - down_n, "вниз", "остаётся"))
                zone.append(pd.Series(z, index=g.index))
            grouped["zone"] = pd.concat(zone).sort_index()
            grouped["group_size"] = gsize
            moves.append(grouped)

        step = pd.concat(moves, ignore_index=True)

        # нулевая активность — понижение независимо от места
        step.loc[step["active_days"] == 0, "zone"] = "вниз"

        new_league = step["league"].copy()
        new_league += (step["zone"] == "вверх").astype(int)
        new_league -= (step["zone"] == "вниз").astype(int)
        new_league = new_league.clip(0, len(LEAGUES) - 1)
        step["new_league"] = new_league

        # сезонное окно: знак за месяц без понижения
        sign = (step["zone"] != "вниз").astype(int)
        step["window"] = [w[-5:] + [s] for w, s in zip(step["window"], sign)]
        step["season"] = [sum(w) for w in step["window"]]

        # паузы: одна за 12 активных недель подряд
        step["streak_for_pause"] = step["streak_for_pause"] + step["active_weeks"]
        earned = (step["streak_for_pause"] >= 12) & (step["pauses"] < 2)
        step.loc[earned, "pauses"] += 1
        step.loc[earned, "streak_for_pause"] -= 12

        history.append(step[["profile_id", "month", "league", "new_league", "zone",
                             "points", "active_days", "capped_sum", "season",
                             "pauses", "group_id", "group_size"]].copy())

        state = step[["profile_id", "new_league", "season", "window",
                      "pauses", "streak_for_pause"]].rename(columns={"new_league": "league"})

    return pd.concat(history, ignore_index=True), state

sim, final_state = run_simulation(mp)
print(f"строк прогона: {len(sim):,}   циклов: {sim['month'].nunique()}")
sim.head()

строк прогона: 39,610   циклов: 5


,profile_id,month,league,new_league,zone,points,active_days,capped_sum,season,pauses,group_id,group_size
0,U002060,2026-04,0,0,остаётся,129.3,3.0,691.92,1,0,L0-00000,13
1,U003162,2026-04,0,1,вверх,482.9,10.0,3627.11,1,0,L0-00000,13
2,U000581,2026-04,0,0,остаётся,382.6,8.0,1614.71,1,0,L0-00000,13
3,U001283,2026-04,0,1,вверх,458.4,9.0,4140.01,1,0,L0-00000,13
4,U006678,2026-04,0,0,остаётся,311.0,5.0,1410.01,1,0,L0-00000,13


In [15]:
dist = (sim.assign(лига=sim["new_league"].map(dict(enumerate(LEAGUE_NAMES))))
          .pivot_table(index="лига", columns="month", values="profile_id",
                       aggfunc="count", fill_value=0))
dist = dist.reindex(LEAGUE_NAMES).fillna(0).astype(int)
print("Распределение участников по лигам на конец каждого месяца:")
print(dist.to_string())

print()
print("Доля зон за весь прогон:")
print((sim["zone"].value_counts(normalize=True) * 100).round(1).to_string())

Распределение участников по лигам на конец каждого месяца:
month         2026-04  2026-05  2026-06  2026-07  2026-08
лига                                                     
Бронзовая        1016      955      914      896      884
Серебряная       1243     1296     1330     1353     1384
Золотая          1199     1273     1356     1431     1484
Платиновая       1216     1299     1373     1429     1469
Рубиновая        1181     1249     1240     1214     1193
Изумрудная       1212     1035      950      879      818
Сапфировая        652      589      516      471      440
Аметистовая       203      186      188      184      179
Обсидиановая        0       40       48       54       57
Жемчужная           0        0        7       10       13
Алмазная            0        0        0        1        1

Доля зон за весь прогон:
zone
остаётся    51.1
вниз        26.9
вверх       22.1


In [16]:
top = sim[sim["league"] >= 8]
if len(top):
    churn = (top["zone"] == "вниз").mean()
    print(f"Вылет из трёх верхних лиг: {100*churn:.1f} % случаев")
    stay_len = (sim.sort_values(["profile_id", "month"])
                  .assign(hi=lambda d: d["league"] >= 8)
                  .groupby("profile_id")["hi"].sum())
    print(f"Покупателей, побывавших в верхних лигах: {(stay_len > 0).sum():,}")
    print(f"Из них продержались больше одного месяца: {(stay_len > 1).sum():,}")
else:
    print("В верхние лиги за прогон никто не дошёл")

print()
print("Сезонное окно на последнем месяце (знаков из 6):")
last = sim[sim["month"] == sim["month"].max()]
print(last["season"].value_counts().sort_index().to_string())
print()
print("Паузы в запасе:")
print(last["pauses"].value_counts().sort_index().to_string())

Вылет из трёх верхних лиг: 56.2 % случаев
Покупателей, побывавших в верхних лигах: 106
Из них продержались больше одного месяца: 39

Сезонное окно на последнем месяце (знаков из 6):
season
0       4
1      85
2     859
3    2310
4    3096
5    1568

Паузы в запасе:
pauses
0     609
1    5681
2    1632


## 5. Экономика — проверка самого рискованного предположения

Проверяем не «работает ли», а **где перестаёт работать**. Допущения фиксируются здесь,
до расчёта, и меняются в одном месте.

Логика: награда — кэшбэк на выбранные категории с месячным потолком, плюс скидка на любимые
товары. Прирост маржи считается от инкрементального оборота, который механика создаёт.
Условие приемлемости из кейса: **маржа на участника не снижается**.

In [17]:
ECON = {
    "BASE_MARGIN": 0.22,        # валовая маржа сети, доля от оборота
    "CASHBACK_RATE": 0.05,      # ставка кэшбэка на выбранные категории
    "CAT_SHARE_PER_CAT": 0.028, # доля оборота в одной выбранной подкатегории (дробные категории)
    "BASE_CAP_RUB": 300,        # базовый месячный потолок начисления, ×множитель лиги
    "FAV_COST_RUB": 55,         # стоимость скидки на один любимый товар в месяц
    "UPLIFT_BY_LEAGUE": [0.02, 0.025, 0.03, 0.035, 0.04, 0.045,
                         0.05, 0.055, 0.06, 0.065, 0.07],  # прирост оборота, доля
}
# пакет условий по лигам: подкатегорий, множитель лимита, любимых товаров
# категории дробные (2–3 % оборота каждая), любимые товары открываются с рубиновой лиги
PACKS = [(4, 1.0, 0), (4, 1.1, 0), (5, 1.2, 0), (5, 1.3, 0), (6, 1.4, 1), (6, 1.5, 1),
         (7, 1.6, 2), (8, 1.8, 2), (8, 2.0, 3), (9, 2.2, 3), (10, 2.5, 4)]

def reward_cost(spend, league, econ=ECON):
    cats, limit_k, favs = PACKS[league]
    eligible = spend * min(1.0, cats * econ["CAT_SHARE_PER_CAT"])
    cashback = min(eligible * econ["CASHBACK_RATE"], econ["BASE_CAP_RUB"] * limit_k)
    return cashback + favs * econ["FAV_COST_RUB"]

def economics(sim, econ=ECON):
    d = sim.copy()
    d["spend"] = d["capped_sum"]
    d["uplift"] = d["new_league"].map(lambda l: econ["UPLIFT_BY_LEAGUE"][l])
    d["incremental_spend"] = d["spend"] * d["uplift"]
    d["incremental_margin"] = d["incremental_spend"] * econ["BASE_MARGIN"]
    d["cost"] = [reward_cost(s, l, econ) for s, l in zip(d["spend"], d["new_league"])]
    d["net"] = d["incremental_margin"] - d["cost"]
    return d

ec = economics(sim)
by_league = ec.groupby("new_league").agg(
    участников=("profile_id", "count"),
    оборот=("spend", "mean"),
    прирост_маржи=("incremental_margin", "mean"),
    стоимость_награды=("cost", "mean"),
    итог=("net", "mean"),
).round(1)
by_league.index = [LEAGUE_NAMES[i] for i in by_league.index]
print("На одного участника в месяц, ₽:")
print(by_league.to_string())
print(f"\nИтог по всей базе на участника в месяц: {ec['net'].mean():.1f} ₽")
print(f"Доля участников с отрицательным итогом: {100*(ec['net'] < 0).mean():.1f} %")

На одного участника в месяц, ₽:
              участников   оборот  прирост_маржи  стоимость_награды  итог
Бронзовая           4665   1660.4            7.3                9.3  -2.0
Серебряная          6606   3519.5           19.4               19.7  -0.4
Золотая             6743   5059.8           33.4               35.4  -2.0
Платиновая          6786   6974.5           53.7               48.8   4.9
Рубиновая           6077   9401.3           82.7              134.0 -51.2
Изумрудная          4894  12746.8          126.2              162.1 -35.9
Сапфировая          2668  18378.3          202.2              290.1 -87.9
Аметистовая          940  25662.4          310.5              396.5 -85.9
Обсидиановая         199  32960.2          435.1              533.6 -98.5
Жемчужная             30  43619.8          623.8              688.1 -64.3
Алмазная               2  55645.2          856.9              937.9 -81.0

Итог по всей базе на участника в месяц: -20.6 ₽
Доля участников с отрицательным

In [18]:
# Граница окупаемости: при каком приросте оборота конструкция выходит в ноль
def breakeven_uplift(league, econ=ECON, spend=None):
    if spend is None:
        spend = ec.loc[ec["new_league"] == league, "spend"].mean()
    if not np.isfinite(spend) or spend <= 0:
        return np.nan
    cost = reward_cost(spend, league, econ)
    return cost / (spend * econ["BASE_MARGIN"])

rows = []
for l in range(len(LEAGUES)):
    sub = ec[ec["new_league"] == l]
    if len(sub) == 0:
        continue
    be = breakeven_uplift(l)
    rows.append({"лига": LEAGUE_NAMES[l],
                 "средний оборот, ₽": round(sub["spend"].mean()),
                 "нужен прирост, %": round(100 * be, 1),
                 "заложен прирост, %": round(100 * ECON["UPLIFT_BY_LEAGUE"][l], 1),
                 "запас, п.п.": round(100 * (ECON["UPLIFT_BY_LEAGUE"][l] - be), 1)})
be_df = pd.DataFrame(rows)
print("Граница окупаемости по лигам (при марже "
      f"{100*ECON['BASE_MARGIN']:.0f} %):")
print(be_df.to_string(index=False))

Граница окупаемости по лигам (при марже 22 %):
        лига  средний оборот, ₽  нужен прирост, %  заложен прирост, %  запас, п.п.
   Бронзовая               1660               2.5                 2.0         -0.5
  Серебряная               3519               2.5                 2.5         -0.0
     Золотая               5060               3.2                 3.0         -0.2
  Платиновая               6975               3.2                 3.5          0.3
   Рубиновая               9401               6.5                 4.0         -2.5
  Изумрудная              12747               5.8                 4.5         -1.3
  Сапфировая              18378               7.2                 5.0         -2.2
 Аметистовая              25662               7.0                 5.5         -1.5
Обсидиановая              32960               7.4                 6.0         -1.4
   Жемчужная              43620               7.4                 6.5         -0.9
    Алмазная              55645         

In [19]:
# Карта устойчивости: где решение уходит в минус при разных марже и отклике
margins = [0.16, 0.18, 0.20, 0.22, 0.25, 0.28]
scales = [0.5, 0.75, 1.0, 1.25, 1.5]   # множитель к заложенному приросту

grid = pd.DataFrame(index=[f"×{s}" for s in scales],
                    columns=[f"{int(100*m)} %" for m in margins], dtype=float)
for s in scales:
    for m in margins:
        econ = dict(ECON)
        econ["BASE_MARGIN"] = m
        econ["UPLIFT_BY_LEAGUE"] = [u * s for u in ECON["UPLIFT_BY_LEAGUE"]]
        e = economics(sim, econ)
        grid.loc[f"×{s}", f"{int(100*m)} %"] = round(e["net"].mean(), 1)

print("Итог на участника в месяц, ₽. Строки — множитель отклика, столбцы — валовая маржа:")
print(grid)
print()
print("Отрицательные ячейки — зона, где награда съедает эффект.")

Итог на участника в месяц, ₽. Строки — множитель отклика, столбцы — валовая маржа:
       16 %  18 %  20 %  22 %  25 %  28 %
×0.5  -65.8 -62.5 -59.3 -56.1 -51.2 -46.4
×0.75 -52.9 -48.0 -43.2 -38.3 -31.1 -23.8
×1.0  -40.0 -33.5 -27.1 -20.6 -10.9  -1.3
×1.25 -27.1 -19.0 -10.9  -2.9   9.2  21.3
×1.5  -14.2  -4.5   5.2  14.9  29.4  43.9

Отрицательные ячейки — зона, где награда съедает эффект.


In [14]:
# Обратная задача: при каких параметрах награды конструкция выходит в ноль
def net_at(rate=None, cap=None, fav=None, econ=ECON):
    e = dict(econ)
    if rate is not None: e["CASHBACK_RATE"] = rate
    if cap is not None:  e["BASE_CAP_RUB"] = cap
    if fav is not None:  e["FAV_COST_RUB"] = fav
    return economics(sim, e)["net"].mean()

rows = []
for rate in [0.05, 0.04, 0.03, 0.02, 0.015, 0.01, 0.005]:
    rows.append({"ставка кэшбэка, %": round(100*rate, 1),
                 "итог на участника, ₽": round(net_at(rate=rate), 1)})
print("Что даёт снижение ставки кэшбэка при прочих равных:")
print(pd.DataFrame(rows).to_string(index=False))

rows = []
for cap in [300, 200, 150, 100, 60, 30]:
    rows.append({"базовый потолок, ₽": cap,
                 "итог на участника, ₽": round(net_at(cap=cap), 1)})
print()
print("Что даёт снижение потолка начисления:")
print(pd.DataFrame(rows).to_string(index=False))

Что даёт снижение ставки кэшбэка при прочих равных:
 ставка кэшбэка, %  итог на участника, ₽
               5.0                 -20.6
               4.0                  -7.6
               3.0                   5.5
               2.0                  18.6
               1.5                  25.1
               1.0                  31.7
               0.5                  38.2

Что даёт снижение потолка начисления:
 базовый потолок, ₽  итог на участника, ₽
                300                 -20.6
                200                 -20.0
                150                 -18.4
                100                 -13.5
                 60                  -3.3
                 30                  13.3


### Что показал расчёт

При заявленных допущениях конструкция **в минусе на всех лигах**. Причина структурная,
и её видно в таблице границы окупаемости: кэшбэк платится со **всего** оборота участника,
а прирост маржи возникает только на **инкрементальном обороте**. Чтобы сойтись, нужен прирост
оборота 9–14 %, тогда как в допущениях заложено 2–7 %.

Это и есть ответ на самое рискованное предположение — отрицательный при этих параметрах.
Варианта два: либо награда должна быть заметно дешевле (таблицы выше показывают, насколько),
либо начисление привязывается не ко всему обороту, а к приросту над личной базой участника.
Второй вариант меняет саму механику награды и требует отдельного решения.

## 5б. Ставка кэшбэка по маржинальности группы

Единая ставка игнорирует то, что группы товаров зарабатывают по-разному. Привязка ставки к марже группы выравнивает это: награда всегда
составляет одну и ту же долю от заработанного на этой группе.

In [20]:
# Допущение: валовая маржа по товарным группам. Заменить на реальные цифры X5, когда появятся.
CAT_MARGIN = {
    "готовая еда": 0.38, "бытовая химия": 0.35, "кофе и чай": 0.32, "сладости": 0.30,
    "овощи и фрукты": 0.28, "заморозка": 0.26, "напитки": 0.25, "алкоголь": 0.24,
    "хлеб": 0.22, "мясо и птица": 0.20, "бакалея": 0.20, "молочное": 0.18,
    "детское питание": 0.15, "табак": 0.06,
}
RATE_K = 0.25                      # ставка кэшбэка = доля от маржи группы
RATE_MIN, RATE_MAX = 0.01, 0.08    # ставка не выходит за эти границы
SUB_SPLIT = [0.45, 0.33, 0.22]     # дробление группы на три подкатегории

CAT_RATE = {k: float(np.clip(RATE_K * m, RATE_MIN, RATE_MAX)) for k, m in CAT_MARGIN.items()}
tbl = pd.DataFrame({"маржа группы, %": pd.Series(CAT_MARGIN) * 100,
                    "ставка кэшбэка, %": pd.Series(CAT_RATE) * 100})
tbl["единая ставка 5 % съедает маржи, %"] = (0.05 / pd.Series(CAT_MARGIN) * 100).round(0)
tbl["ставка по марже съедает, %"] = (pd.Series(CAT_RATE) / pd.Series(CAT_MARGIN) * 100).round(0)
print(tbl.sort_values("маржа группы, %").round(1).to_string())

                 маржа группы, %  ставка кэшбэка, %  единая ставка 5 % съедает маржи, %  ставка по марже съедает, %
табак                        6.0                1.5                                83.0                        25.0
детское питание             15.0                3.8                                33.0                        25.0
молочное                    18.0                4.5                                28.0                        25.0
мясо и птица                20.0                5.0                                25.0                        25.0
бакалея                     20.0                5.0                                25.0                        25.0
хлеб                        22.0                5.5                                23.0                        25.0
алкоголь                    24.0                6.0                                21.0                        25.0
напитки                     25.0                6.2                     

In [22]:
# раскладываем зачтённые чеки по категориям
q = rq[rq["qualified"]][["profile_id", "month", "categories"]]
rows = []
for pid, m, cats in q.itertuples(index=False):
    for part in cats.split("|"):
        k, v = part.rsplit(":", 1)
        rows.append((pid, m, k, float(v)))
cat = pd.DataFrame(rows, columns=["profile_id", "month", "category", "amount"])
cat = cat.groupby(["profile_id", "month", "category"], as_index=False)["amount"].sum()

# средневзвешенная маржа корзины вместо плоской
cat["margin_rub"] = cat["amount"] * cat["category"].map(CAT_MARGIN)
basket = cat.groupby(["profile_id", "month"]).agg(
    spend_cat=("amount", "sum"), margin_rub=("margin_rub", "sum")).reset_index()
basket["basket_margin"] = basket["margin_rub"] / basket["spend_cat"]
print(f"Строк «профиль × месяц × категория»: {len(cat):,}")
print(f"Средневзвешенная маржа корзины: {basket['basket_margin'].mean():.3f} "
      f"против плоского допущения {ECON['BASE_MARGIN']}")

# дробим группы и оставляем лучшие подкатегории по тратам участника
sub = cat.loc[cat.index.repeat(len(SUB_SPLIT))].copy()
sub["sub_idx"] = np.tile(np.arange(len(SUB_SPLIT)), len(cat))
sub["amount"] = sub["amount"] * np.array(SUB_SPLIT)[sub["sub_idx"]]
sub["rate"] = sub["category"].map(CAT_RATE)
sub = sub.sort_values(["profile_id", "month", "amount"], ascending=[True, True, False])
sub["rank"] = sub.groupby(["profile_id", "month"]).cumcount() + 1

Строк «профиль × месяц × категория»: 489,597
Средневзвешенная маржа корзины: 0.251 против плоского допущения 0.22


In [23]:
ec2 = economics(sim)
key = pd.MultiIndex.from_arrays([ec2["profile_id"], ec2["month"]])
slots = np.array([PACKS[l][0] for l in ec2["new_league"]])
favs  = np.array([PACKS[l][2] for l in ec2["new_league"]])
limit = np.array([PACKS[l][1] for l in ec2["new_league"]])

sub["slots"] = pd.Series(slots, index=key).reindex(
    pd.MultiIndex.from_arrays([sub["profile_id"], sub["month"]])).values
chosen = sub[sub["rank"] <= sub["slots"].fillna(0)]

def cashback(rate_map):
    return (chosen.assign(cb=chosen["amount"] * chosen["category"].map(rate_map))
                  .groupby(["profile_id", "month"])["cb"].sum().reindex(key).fillna(0).values)

ec2["basket_margin"] = basket.set_index(["profile_id", "month"])["basket_margin"]                              .reindex(key).fillna(ECON["BASE_MARGIN"]).values
ec2["margin_real"] = ec2["spend"] * ec2["uplift"] * ec2["basket_margin"]
ec2["cap"] = ECON["BASE_CAP_RUB"] * limit
ec2["fav"] = favs * ECON["FAV_COST_RUB"]

# начисление на прирост над личной базой
mps = mp.sort_values(["profile_id", "month"])
mps["baseline"] = mps.groupby("profile_id")["capped_sum"].transform(
    lambda s: s.shift(1).expanding().mean())
bl = mps.set_index(["profile_id", "month"])["baseline"].reindex(key).values
ec2["baseline"] = np.where(np.isnan(bl), ec2["spend"], bl)
ec2["excess_share"] = np.where(ec2["spend"] > 0,
                               (ec2["spend"] - ec2["baseline"]).clip(lower=0) / ec2["spend"], 0)

flat = {k: ECON["CASHBACK_RATE"] for k in CAT_MARGIN}
def summarize(cost, margin, label):
    net = margin - cost
    return {"вариант": label, "награда, ₽": round(np.mean(cost), 1),
            "итог, ₽": round(np.mean(net), 1),
            "в минусе, %": round(100 * np.mean(net < 0), 1)}

res = [
    summarize(np.minimum(cashback(flat), ec2["cap"]) + ec2["fav"],
              ec2["incremental_margin"], "единая ставка, плоская маржа"),
    summarize(np.minimum(cashback(flat), ec2["cap"]) + ec2["fav"],
              ec2["margin_real"], "единая ставка, маржа по корзине"),
    summarize(np.minimum(cashback(CAT_RATE), ec2["cap"]) + ec2["fav"],
              ec2["margin_real"], "ставка = четверть маржи группы"),
]
for k in (0.15, 0.10):
    rm = {kk: float(np.clip(k * m, RATE_MIN, RATE_MAX)) for kk, m in CAT_MARGIN.items()}
    res.append(summarize(np.minimum(cashback(rm), ec2["cap"]) + ec2["fav"],
                         ec2["margin_real"], f"ставка = {int(k*100)} % маржи группы"))
for k in (0.25, 0.15):
    rm = {kk: float(np.clip(k * m, RATE_MIN, RATE_MAX)) for kk, m in CAT_MARGIN.items()}
    cost = np.minimum(cashback(rm) * ec2["excess_share"], ec2["cap"]) + ec2["fav"]
    res.append(summarize(cost, ec2["margin_real"],
                         f"{int(k*100)} % маржи группы + начисление на прирост"))
print("На одного участника в месяц:")
print(pd.DataFrame(res).to_string(index=False))

На одного участника в месяц:
                                  вариант  награда, ₽  итог, ₽  в минусе, %
             единая ставка, плоская маржа       180.1   -109.2         99.0
          единая ставка, маржа по корзине       180.1   -100.4         98.9
           ставка = четверть маржи группы       203.1   -123.4         98.9
               ставка = 15 % маржи группы       141.2    -61.5         98.6
               ставка = 10 % маржи группы       103.8    -24.1         86.6
25 % маржи группы + начисление на прирост        56.9     22.8         23.9
15 % маржи группы + начисление на прирост        45.1     34.6         11.8


### Что даёт привязка к марже

Сама по себе она уровень затрат не снижает: коэффициент задаёт средний уровень ставки, и при
четверти маржи средняя ставка выходит выше прежних 5 %. Ценность в другом — в **гарантии**.
Стоимость награды на любой группе теперь равна фиксированной доле от маржи, которую эта группа
приносит. Кэшбэк физически не может съесть больше заданной части заработанного, чего единая
ставка не обеспечивает: на табаке она забирала 83 % маржи, на детском питании — треть.

Попутно выяснилось, что плоское допущение о марже 22 % занижало прирост: средневзвешенная
маржа корзины на этих данных выше. Прирост маржи в расчётах вырос примерно на десятую часть.

Уровень затрат по-прежнему задаётся базой начисления, а не ставкой. Связка «ставка по марже
плюс начисление на прирост над личной базой» — единственная конфигурация, которая выходит в плюс.

## 5в. Зафиксированная конструкция награды

Итог подбора: параметры, при которых сеть выходит ровно в ноль, а покупатель получает максимум.
Начисление — со всего оборота. Табак и алкоголь исключены: стимулирование их продаж ограничено законом.

| Рычаг | Значение |
|---|---|
| Дробление групп | на 6 равных подкатегорий |
| Групп на выбор (пул) | 4 → 10 по лигам |
| Подкатегорий можно выбрать (слоты) | 3 → 7 по лигам |
| Ставка | 0,197 × маржа группы, одна для всех лиг |
| Любимые товары | не больше 4, с рубиновой лиги |

In [25]:
FIXED = {
    "K": 6,                                                    # подкатегорий в группе, равными долями
    "pool":  [4, 5, 6, 6, 7, 7, 8, 9, 9, 10, 10],              # групп доступно для выбора
    "slots": [3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 7],                # подкатегорий можно выбрать
    "favs":  [0, 0, 0, 0, 1, 1, 2, 2, 3, 3, 4],                # любимых товаров
    "k_base": 0.197,                                           # доля маржи в ставке, одна для всех лиг
    "excluded": ["табак", "алкоголь"],
}

CM = {c: m for c, m in CAT_MARGIN.items() if c not in FIXED["excluded"]}
cat_f = cat[cat["category"].isin(CM)].copy()

ecf = economics(sim)
keyf = pd.MultiIndex.from_arrays([ecf["profile_id"], ecf["month"]])
lg = ecf["new_league"].values
lim = np.array([PACKS[l][1] for l in lg])
ecf["basket_margin"] = basket.set_index(["profile_id", "month"])["basket_margin"]                              .reindex(keyf).fillna(ECON["BASE_MARGIN"]).values
margin_f = (ecf["spend"] * ecf["uplift"] * ecf["basket_margin"]).values

# группы открываются от наименее ходовых к самым ходовым
pop = cat_f.groupby("category")["amount"].sum().sort_values(ascending=False).index.tolist()
rank_f = {c: i for i, c in enumerate(pop[::-1])}

w = np.ones(FIXED["K"]) / FIXED["K"]
sf = cat_f.loc[cat_f.index.repeat(FIXED["K"])].copy()
sf["amount"] = sf["amount"].values * np.tile(w, len(cat_f))
sf["cat_rank"] = sf["category"].map(rank_f)
k2 = pd.MultiIndex.from_arrays([sf["profile_id"], sf["month"]])
sf["league"] = pd.Series(lg, index=keyf).reindex(k2).values
sf = sf[sf["league"].notna()].copy(); sf["league"] = sf["league"].astype(int)

sf = sf[sf["cat_rank"] < np.array(FIXED["pool"])[sf["league"]]]
sf = sf.sort_values(["profile_id", "month", "amount"], ascending=[True, True, False])
sf["rank"] = sf.groupby(["profile_id", "month"]).cumcount() + 1
pick = sf[sf["rank"] <= np.array(FIXED["slots"])[sf["league"]]].copy()

pick["rate"] = (FIXED["k_base"] * pick["category"].map(CM)).clip(0.002, 0.30)
cb_f = pick.assign(cb=pick["amount"] * pick["rate"])            .groupby(["profile_id", "month"])["cb"].sum().reindex(keyf).fillna(0).values
cov_f = pick.groupby(["profile_id", "month"])["amount"].sum().reindex(keyf).fillna(0).values         / np.maximum(ecf["spend"].values, 1)
cost_f = np.minimum(cb_f, ECON["BASE_CAP_RUB"] * lim) + np.array(FIXED["favs"])[lg] * ECON["FAV_COST_RUB"]
net_f = margin_f - cost_f

print(f"Награда: {cost_f.mean():.1f} ₽ на участника в месяц")
print(f"Прирост маржи: {margin_f.mean():.1f} ₽")
print(f"Итог сети: {net_f.mean():.2f} ₽")
print(f"Покрытие корзины: {100 * cov_f.mean():.1f} %")
print(f"Медиана награды: {np.median(cost_f):.1f} ₽, меньше 20 ₽ получают {100 * (cost_f < 20).mean():.1f} %")

Награда: 79.8 ₽ на участника в месяц
Прирост маржи: 79.7 ₽
Итог сети: -0.06 ₽
Покрытие корзины: 12.3 %
Медиана награды: 41.9 ₽, меньше 20 ₽ получают 29.5 %


In [26]:
t = pd.DataFrame({"лига": [LEAGUE_NAMES[l] for l in lg], "оборот": ecf["spend"].values,
                  "прирост маржи": margin_f, "награда": cost_f,
                  "покрытие, %": cov_f * 100, "итог": net_f})
by = t.groupby("лига").mean(numeric_only=True).round(1)
by = by.reindex([n for n in LEAGUE_NAMES if n in by.index])
print("По лигам:")
print(by.to_string())

rate_tbl = pd.DataFrame({
    "маржа группы, %": {c: round(100 * m, 1) for c, m in CM.items()},
    "ставка кэшбэка, %": {c: round(100 * float(np.clip(FIXED["k_base"] * m, 0.002, 0.30)), 1)
                          for c, m in CM.items()},
}).sort_values("маржа группы, %", ascending=False)
print()
print("Ставка по группам (одна для всех лиг):")
print(rate_tbl.to_string())

По лигам:
               оборот  прирост маржи  награда  покрытие, %   итог
лига                                                             
Бронзовая      1660.4            8.5      8.6          8.9   -0.2
Серебряная     3519.5           22.3     18.5          9.7    3.8
Золотая        5059.8           38.2     34.4         12.7    3.7
Платиновая     6974.5           60.8     41.9         11.6   18.8
Рубиновая      9401.3           93.2    121.2         14.6  -28.0
Изумрудная    12746.8          141.0    136.4         13.6    4.6
Сапфировая    18378.3          225.3    258.3         16.5  -33.0
Аметистовая   25662.4          345.5    301.9         16.5   43.6
Обсидиановая  32960.2          481.3    427.7         18.6   53.7
Жемчужная     43619.8          696.2    552.3         19.4  143.9
Алмазная      55645.2          908.8    542.6         18.1  366.2

Ставка по группам (одна для всех лиг):
                 маржа группы, %  ставка кэшбэка, %
готовая еда                 38.0        

Конструкция награды выведена ровно в ноль: награда равна приросту маржи. Это граница, а не цель с
запасом — критерий «без снижения маржи на участника» выполняется на грани, и ошибка в допущении
о приросте оборота уводит в минус. Для пилота разумно целиться в 70–80 % этого бюджета.

Награда концентрируется наверху: бронзовая лига получает около 9 ₽ в месяц, алмазная — свыше
500 ₽, и почти треть участников получает меньше 20 ₽. Выравнивать это наклоном ставки по лигам
мы не стали — наклон разворачивает ставку против лестницы и на экране читается как ухудшение
условий с ростом лиги. Если разрыв окажется проблемой, ровнять его лимитом и числом слотов.

## 6. Антифрод

Семь признаков, каждый формулируется описанными правилами, а не весом в модели. Порог — число сработавших
правил. Метрика — precision на разметке из генератора: ложное лишение лиги дороже
пропущенного накрутчика.

In [31]:
FRAUD = {
    "MAX_RECEIPTS_PER_DAY": 3,     # больше — подозрительно
    "SMALL_SHARE": 0.55,           # доля чеков ниже порога зачёта
    "TIGHT_GAP_SHARE": 0.10,       # доля чеков с интервалом меньше минимального
    "ACTIVE_DAYS_MONTH": 26,       # активных дней в месяце
    "SINGLE_STORE": 0.98,          # доля чеков в одном магазине
    "SYNC_SHARE": 0.06,            # доля чеков, совпавших с другим профилем по магазину, дате и часу
    "TRIGGER_THRESHOLD": 2,        # сработавших правил для блокировки
}

r = rq.copy()
r["small"] = r["amount_rub"] < POINTS["MIN_RECEIPT_RUB"]
r["tight"] = r["gap_min"].notna() & (r["gap_min"] < POINTS["MIN_GAP_MIN"])
r["hour"] = r["ts"].dt.hour

# синхронность: магазин + дата + час, где встретилось больше одного профиля
slot = r.groupby(["store_id", "date", "hour"])["profile_id"].transform("nunique")
r["sync"] = slot > 1

per_day = r.groupby(["profile_id", "date"]).agg(
    n=("receipt_id", "count"), cities=("city", "nunique")).reset_index()

feat = pd.DataFrame({"profile_id": profiles["profile_id"]}).set_index("profile_id")
feat["max_receipts_day"] = per_day.groupby("profile_id")["n"].max()
feat["days_multi_city"] = per_day.assign(x=per_day["cities"] > 1).groupby("profile_id")["x"].sum()
feat["small_share"] = r.groupby("profile_id")["small"].mean()
feat["tight_share"] = r.groupby("profile_id")["tight"].mean()
feat["single_store_share"] = r.groupby("profile_id")["store_id"].apply(
    lambda s: s.value_counts(normalize=True).max())
feat["sync_share"] = r.groupby("profile_id")["sync"].mean()
feat["max_active_days"] = mp.groupby("profile_id")["active_days"].max()
feat = feat.fillna(0)
feat.describe().round(2)

,max_receipts_day,days_multi_city,small_share,tight_share,single_store_share,sync_share,max_active_days
count,8000.00,8000.00,8000.00,8000.00,8000.00,8000.00,8000.00
mean,1.05,0.06,0.01,0.01,0.73,0.08,12.18
std,0.36,0.95,0.07,0.05,0.25,0.10,5.89
min,1.00,0.00,0.00,0.00,0.31,0.00,0.00
25%,1.00,0.00,0.00,0.00,0.52,0.00,8.00
50%,1.00,0.00,0.00,0.00,0.62,0.05,11.00
75%,1.00,0.00,0.00,0.00,1.00,0.13,15.00
max,4.00,29.00,1.00,0.59,1.00,0.68,31.00


In [33]:
rules = pd.DataFrame(index=feat.index)
rules["много чеков за день"]        = feat["max_receipts_day"] > FRAUD["MAX_RECEIPTS_PER_DAY"]
rules["чеки вплотную друг к другу"] = feat["tight_share"] > FRAUD["TIGHT_GAP_SHARE"]
rules["много мелких чеков"]         = feat["small_share"] > FRAUD["SMALL_SHARE"]
rules["ходит каждый день"]          = feat["max_active_days"] >= FRAUD["ACTIVE_DAYS_MONTH"]
rules["покупки в двух городах за день"] = feat["days_multi_city"] > 0
rules["всё в одном магазине"]       = feat["single_store_share"] >= FRAUD["SINGLE_STORE"]
rules["синхронен с другими"]        = feat["sync_share"] > FRAUD["SYNC_SHARE"]

score = rules.sum(axis=1)
flag = score >= FRAUD["TRIGGER_THRESHOLD"]

truth = profiles.set_index("profile_id")["is_fraud"].astype(bool).reindex(rules.index).fillna(False)
ftype = profiles.set_index("profile_id")["fraud_type"].reindex(rules.index).fillna("")

tp = int((flag & truth).sum()); fp = int((flag & ~truth).sum())
fn = int((~flag & truth).sum()); tn = int((~flag & ~truth).sum())
precision = tp / (tp + fp) if tp + fp else float("nan")
recall = tp / (tp + fn) if tp + fn else float("nan")

print(f"Порог: сработало правил ≥ {FRAUD['TRIGGER_THRESHOLD']}")
print(f"Помечено профилей: {int(flag.sum()):,} из {len(flag):,}")
print(f"precision: {precision:.3f}   recall: {recall:.3f}")
print(f"TP {tp}   FP {fp}   FN {fn}   TN {tn}")

Порог: сработало правил ≥ 2
Помечено профилей: 1,927 из 8,000
precision: 0.139   recall: 0.761
TP 268   FP 1659   FN 84   TN 5989


In [34]:
print("Точность каждого правила по отдельности:")
per_rule = []
for name in rules.columns:
    f = rules[name]
    p = (f & truth).sum() / f.sum() if f.sum() else float("nan")
    per_rule.append({"правило": name, "сработало": int(f.sum()),
                     "из них накрутчики": int((f & truth).sum()),
                     "precision": round(p, 3) if f.sum() else None})
print(pd.DataFrame(per_rule).to_string(index=False))

print()
print("Ловим ли каждый тип накрутки:")
rec_by_type = []
for t in sorted(x for x in ftype.unique() if x):
    mask = ftype == t
    rec_by_type.append({"тип": t, "профилей": int(mask.sum()),
                        "поймано": int((flag & mask).sum()),
                        "recall": round((flag & mask).sum() / mask.sum(), 3)})
print(pd.DataFrame(rec_by_type).to_string(index=False))

Точность каждого правила по отдельности:
                       правило  сработало  из них накрутчики  precision
           много чеков за день        108                108      1.000
    чеки вплотную друг к другу        112                112      1.000
            много мелких чеков         46                 46      1.000
             ходит каждый день        280                 17      0.061
покупки в двух городах за день         48                 48      1.000
          всё в одном магазине       3564                172      0.048
           синхронен с другими       3583                199      0.056

Ловим ли каждый тип накрутки:
                    тип  профилей  поймано  recall
        дробление чеков       112      110   0.982
     мелкие чеки подряд       104       47   0.452
несовместимая география        48       24   0.500
        сговор в группе        88       87   0.989


In [35]:
print("Как порог влияет на precision и recall:")
rows = []
for t in range(1, len(rules.columns) + 1):
    f = score >= t
    tp_ = int((f & truth).sum()); fp_ = int((f & ~truth).sum()); fn_ = int((~f & truth).sum())
    rows.append({"порог правил": t, "помечено": int(f.sum()),
                 "precision": round(tp_ / (tp_ + fp_), 3) if tp_ + fp_ else None,
                 "recall": round(tp_ / (tp_ + fn_), 3) if tp_ + fn_ else None})
print(pd.DataFrame(rows).to_string(index=False))

Как порог влияет на precision и recall:
 порог правил  помечено  precision  recall
            1      5678      0.059   0.957
            2      1927      0.139   0.761
            3       115      0.661   0.216
            4        20      1.000   0.057
            5         1      1.000   0.003
            6         0        NaN   0.000
            7         0        NaN   0.000


In [36]:
# Правила неравноценны: три из них точные, остальные шумят.
# Собираем набор, где точные срабатывают в одиночку, а шумные — только вместе.
strong = (rules["чеки вплотную друг к другу"]
          | rules["много мелких чеков"]
          | rules["покупки в двух городах за день"]
          | rules["много чеков за день"])
weak = (rules["всё в одном магазине"].astype(int)
        + rules["синхронен с другими"].astype(int)
        + rules["ходит каждый день"].astype(int))

variants = {
    "любое одно правило из семи": score >= 1,
    "два и более правила из семи": score >= 2,
    "три и более правила из семи": score >= 3,
    "только точные правила": strong,
    "точные, либо все три шумных вместе": strong | (weak >= 3),
}
rows = []
for name, f in variants.items():
    tp_ = int((f & truth).sum()); fp_ = int((f & ~truth).sum()); fn_ = int((~f & truth).sum())
    rows.append({"набор правил": name, "помечено": int(f.sum()),
                 "precision": round(tp_ / (tp_ + fp_), 3) if tp_ + fp_ else None,
                 "recall": round(tp_ / (tp_ + fn_), 3) if tp_ + fn_ else None})
print(pd.DataFrame(rows).to_string(index=False))

best = strong   # выбираем набор с максимальной precision: ложное лишение лиги дороже пропуска
print()
print("Выбранный набор — по типам накрутки:")
rows = []
for t in sorted(x for x in ftype.unique() if x):
    mask = ftype == t
    rows.append({"тип": t, "профилей": int(mask.sum()),
                 "поймано": int((best & mask).sum()),
                 "recall": round((best & mask).sum() / mask.sum(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

                      набор правил  помечено  precision  recall
        любое одно правило из семи      5678      0.059   0.957
       два и более правила из семи      1927      0.139   0.761
       три и более правила из семи       115      0.661   0.216
             только точные правила       205      1.000   0.582
точные, либо все три шумных вместе       247      0.842   0.591

Выбранный набор — по типам накрутки:
                    тип  профилей  поймано  recall
        дробление чеков       112      112   1.000
     мелкие чеки подряд       104       45   0.433
несовместимая география        48       48   1.000
        сговор в группе        88        0   0.000


### Формулировка порога словами

Правило, которое можно показать участнику и оспорить:

> Начисление приостанавливается, если выполнено **хотя бы одно** из четырёх условий:
> больше трёх чеков за один день; больше десятой части чеков пробита в одном магазине
> с интервалом меньше сорока минут; больше половины чеков ниже ста рублей; покупки в двух
> городах в один день.

Три оставшихся признака — все покупки в одном магазине, активность почти каждый день,
совпадения с другим участником по магазину и часу — в блокировку **не входят**. Точность
каждого из них около 5 %: на одного накрутчика приходится два десятка обычных покупателей.
Их место — очередь на ручной разбор, а не автоматическое лишение лиги.

Сравнение наборов в таблице выше показывает цену компромисса: добавление мягких признаков
к точным поднимает recall меньше чем на процентный пункт, но роняет precision с 1,00 до 0,84.
При правиле «ложное лишение лиги дороже пропущенного накрутчика» такой обмен невыгоден.

**Чего этот набор не ловит.** Сговор в группе он пропускает полностью: единственные признаки,
которые на него указывают — общий магазин и синхронность, а они как раз шумные. Ловить сговор
автоматически по текущим данным не выйдет, нужен либо более узкий признак (совпадение по
магазину, дате и **минуте**, а не часу), либо ручной разбор. Это же ограничение отмечено
в описании синтетики: вектор сговора смоделирован грубо.

## Что этот прогон не проверяет

- Отклик на механику задан допущением `UPLIFT_BY_LEAGUE`, а не измерен. Карта устойчивости
  показывает, при каком отклике конструкция ломается, но сам отклик — предмет пилота.
- Поведение в данных не реагирует на лигу: человек не начинает ходить чаще, поднявшись выше.
  Обратной связи между игрой и покупками в генераторе нет, поэтому распределение по лигам
  отражает только исходное расслоение.
- Сезонное окно за пять циклов не успевает заполниться до шести знаков.
- Вектор «сговор в группе» в синтетике смоделирован грубо: у участников ровно один общий
  магазин, чего в реальности не бывает. Precision по этому типу завышена.